In [ ]:
# Listening pack: extract exactly 3 clips from the generation tars, plus one
# spectrogram PNG per clip. Panels: (1) the file as generated (mono sum, native
# rate); (2) the scoring front end's input (librosa mono, 22.05 kHz, first 30 s);
# (3) that front-end input after -14 LUFS normalization (the K4 lens).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyloudnorm"], check=True)
import os, glob, tarfile, numpy as np, soundfile as sf, librosa, librosa.display, pyloudnorm as pyln
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
WANT = {"sa3s__rock_L7_num__s1.wav": "audio_C.tar", "ace__rock_L1_verb__s1.wav": "audio_D.tar",
        "mgm__edm_L1_num__s5.wav": "audio_A.tar"}
OUT = "/kaggle/working/listen"; os.makedirs(OUT, exist_ok=True)
found = {}
for t in sorted(glob.glob("/kaggle/input/**/audio_*.tar", recursive=True)):
    with tarfile.open(t) as tf:
        for m in tf.getmembers():
            b = os.path.basename(m.name)
            if m.isfile() and b in WANT and b not in found:
                with open(f"{OUT}/{b}", "wb") as fh: fh.write(tf.extractfile(m).read())
                found[b] = os.path.basename(t)
print(found)
assert set(found) == set(WANT), f"missing: {set(WANT) - set(found)}"
for b, t in WANT.items():
    assert found[b] == t, f"{b} came from {found[b]}, expected {t}"

In [ ]:
def spec(ax, y, sr, title):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=2048, hop_length=512)), ref=np.max)
    img = librosa.display.specshow(S, sr=sr, hop_length=512, x_axis="time", y_axis="log", ax=ax, cmap="magma", vmin=-80, vmax=0)
    ax.set_title(title, fontsize=9); return img
for b in WANT:
    p = f"{OUT}/{b}"; info = sf.info(p)
    x, sr0 = sf.read(p, dtype="float32", always_2d=True); y0 = x.mean(axis=1)
    y, sr = librosa.load(p, sr=22050, mono=True, duration=30)              # scoring front end
    lufs = pyln.Meter(sr).integrated_loudness(y)
    yn = np.clip(y * 10 ** ((-14.0 - lufs) / 20.0), -1, 1)
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
    spec(axes[0], y0, sr0, f"as generated: {info.samplerate} Hz x {info.channels} ch, {info.duration:.2f} s (mono sum shown)")
    spec(axes[1], y, sr, f"scorer input: mono, 22.05 kHz, first 30 s  (integrated loudness {lufs:.1f} LUFS)")
    img = spec(axes[2], yn, sr, "scorer input after -14 LUFS normalization")
    fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.6)
    fig.suptitle(b.replace(".wav", ""), y=0.995)
    fig.savefig(f"{OUT}/{b.replace('.wav', '')}_spectrogram.png", dpi=110, bbox_inches="tight"); plt.close(fig)
print(sorted(os.listdir(OUT)))